# 02 — HOG Feature Extraction

Extracts Histogram of Oriented Gradients (HOG) features from the GTSRB images
and saves the feature matrix + labels for later model training.

**Requires:** notebook 01 to have been run (data in `../data/`).

In [ ]:
# Imports for HOG, image handling, and numerical arrays
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from skimage.feature import hog
from torchvision.datasets import GTSRB

In [ ]:
# Load GTSRB train/test from the data directory (already downloaded in notebook 01)
DATA_DIR = Path("../data")
train_dataset = GTSRB(root=DATA_DIR, split="train", download=False)
test_dataset = GTSRB(root=DATA_DIR, split="test", download=False)
print(f"Train size: {len(train_dataset)}, Test size: {len(test_dataset)}")

In [ ]:
# Display one sample image with its class label
image, label = train_dataset[0]
plt.imshow(image)
plt.title(f"Class {label}: {train_dataset.classes[label]}")
plt.axis("off")
plt.show()

In [ ]:
# Compute HOG on the sample image and check the feature vector length
gray = np.array(image.convert("L"))
features, hog_image = hog(
    gray,
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    transform_sqrt=True,
    visualize=True,
)
print(f"Feature vector length: {features.shape[0]}")

In [ ]:
# Visualize the HOG gradient image next to the original
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(gray, cmap="gray")
axes[0].set_title("Original (grayscale)")
axes[0].axis("off")
axes[1].imshow(hog_image, cmap="gray")
axes[1].set_title("HOG image")
axes[1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Extract HOG features for every image and stack into a feature matrix with labels
def extract_hog(dataset):
    X = []
    y = []
    for i in range(len(dataset)):
        image, label = dataset[i]
        gray = np.array(image.convert("L"))
        features = hog(
            gray,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            transform_sqrt=True,
            visualize=False,
        )
        X.append(features)
        y.append(label)
    return np.array(X), np.array(y)

X_train, y_train = extract_hog(train_dataset)
X_test, y_test = extract_hog(test_dataset)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

In [ ]:
# Save features and labels to .npz for the modeling notebook
np.savez_compressed(DATA_DIR / "hog_features.npz",
                    X_train=X_train, y_train=y_train,
                    X_test=X_test, y_test=y_test)
print(f"Saved to {DATA_DIR / 'hog_features.npz'}")